In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import pipeline
from tqdm import tqdm
from datasets import load_dataset
import pandas as pd

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = False  # Use 4bit quantization to reduce memory usage. Can be False.
load_in_8bit = False  # Use 8bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="azherali/Riazi-8B",  # Choose ANY
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

In [ ]:
def generate_answer(problem):
    messages = [
        {
            "role": "user",
            "content": problem,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )

    # Extract only the generated part
    generated_tokens = outputs[0][inputs["input_ids"].shape[1] :]

    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

In [ ]:
dataset = load_dataset("large-traversaal/mgsm_urdu_final", split="test")
results = []

for row in tqdm(dataset):
    response = generate_answer(row["urdu_question"])  # Your model response

    results.append(
        {
            "question": row["urdu_question"],
            "answer": row["answer_number"],
            "response": response,
        }
    )
    print(row["question"])

df = pd.DataFrame(results)
df.to_csv("Riazi-8B-response.csv", index=False)

## Evaluation using LLM as Judge


In [ ]:
df = pd.read_csv("Riazi-8B-response.csv")

validation_results = []

model_id = "openai/gpt-oss-20b"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
)

for _, row in tqdm(df.iterrows()):

    prompt = f"""You are an answer validator for Urdu math reasoning problems.
    
    Input:
    
    * Question: {row['question']}
    * Solution: {row['response']}
    * Expected Answer: {row['answer']}
    
    Task:
    
    1. Read the Urdu math problem and the provided solution.
    2. Determine the final answer produced by the solution.
    3. Compare it with the Expected Answer.
    4. Treat numerically equivalent values as equal:
    
       * 2 = 2.0 = 2.00
       * 5 = 5.000
       * Ignore insignificant trailing zeros in decimals.
    5. Handle mathematical answers written in LaTeX:
        Extract the mathematical value from LaTeX expressions.
        Treat equivalent forms as equal (e.g., ( \frac{1}{2} ) = 0.5, ( \sqrt{4} ) = 2).
    6. If both answers represent the same numeric value, return:
       True
    7. Otherwise return:
       False
    
    Rules:
    
    * Return ONLY one word: True or False.
    * Do not provide explanations, reasoning, calculations, or extra text.
    * Comparison should be based on the final answer only.
    * For numeric answers, compare by value, not string format.
    * If the solution's final answer cannot be determined with confidence, return False.
     """

    messages = [{"role": "user", "content": prompt}]
    outputs = pipe(messages, max_new_tokens=1000)
    result = (
        True
        if "True.assistantfinalTrue" in outputs[0]["generated_text"][-1]["content"]
        else False
    )
    validation_results.append(result)

df["is_correct"] = validation_results

df.to_csv("Riazi-8B-correct.csv", index=False)

print(df.head())